# Stage 1 & 2 — Baseline Analysis

This notebook validates all baseline implementations using GPT-2 on small samples.
Once the pipeline is confirmed correct, the same code runs on **Mistral-7B** for official experiment results.

---

## What is this project?

We implement an **adaptive KV cache optimization framework** for LLM inference.
During autoregressive decoding, the KV cache grows linearly with sequence length and becomes a major GPU memory bottleneck.
Our method partitions the cache into three tiers (recent / moderate / old) and applies eviction + compression to reduce memory usage while preserving generation quality.

## Why do we need baselines?

Before claiming our method is better, we need to establish what "normal" looks like.
These baselines are the reference points we compare against:

| Baseline | Description |
|----------|-------------|
| **Full cache** | Standard HuggingFace decoding — no eviction, cache grows unbounded |
| **Sliding window** | Keep only the most recent N tokens; evict everything older |
| **Naive truncation** | Hard-cut the prompt to a fixed length before generation begins |

## Evaluation metrics

| Metric | What it measures |
|--------|-----------------|
| `latency_ms_per_token` | How fast each token is generated |
| `throughput_tokens_per_sec` | Tokens generated per second |
| `peak_memory_gb` | Peak GPU memory used by the KV cache |
| `perplexity` | Generation quality — lower is better |

## 1. Setup

Clone the repo, install dependencies, and load the model.

We use **GPT-2** (124M parameters) for validation because it runs fast on any GPU and requires no authorization.
For official experiments, swap `MODEL_NAME` to `mistralai/Mistral-7B-v0.1`.

In [ ]:
# Clone repo and install dependencies (run once per Colab session)
!git clone https://github.com/yanghao13111/adaptive-kv-cache.git 2>/dev/null || echo "Repo already cloned"
%cd adaptive-kv-cache
!git pull
!pip install transformers accelerate datasets -q

In [ ]:
import torch
import sys
sys.path.insert(0, '/content/adaptive-kv-cache')

from transformers import AutoModelForCausalLM, AutoTokenizer

# Load GPT-2 for validation — swap to mistralai/Mistral-7B-v0.1 for official experiments
MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16).to(DEVICE)
model.eval()
model.generation_config.pad_token_id = tokenizer.eos_token_id
print(f"Model loaded: {MODEL_NAME}")

## 2. Validate `metrics.py` — Core Measurement Tools

`metrics.py` provides the two shared measurement functions used by all baselines and our adaptive method.

**`measure_latency()`**
Runs `model.generate()` with warmup steps to stabilize the GPU, then measures wall-clock time and peak GPU memory.
Returns `latency_ms_per_token`, `throughput_tokens_per_sec`, and `peak_memory_gb`.

**`compute_perplexity()`**
Evaluates generation quality on a list of reference texts using a sliding window to handle documents longer than `max_length`.
Returns average perplexity — lower means the model is more confident and the cache is preserving useful context.

In [ ]:
from src.eval.metrics import measure_latency, compute_perplexity

# measure_latency — expects latency_ms_per_token, throughput, peak_memory_gb
latency_result = measure_latency(model, tokenizer, "The quick brown fox", max_new_tokens=50, device=DEVICE)
print("measure_latency() result:")
for k, v in latency_result.items():
    print(f"  {k}: {round(v, 4)}")

In [ ]:
# compute_perplexity — use real varied text to avoid artificially low scores
texts = [
    "The history of artificial intelligence began in antiquity with myths and stories of artificial beings. "
    "The modern field of AI research was founded at a workshop held on the campus of Dartmouth College "
    "during the summer of 1956. The attendees became the founders and leaders of AI research for decades. "
    "Many of them predicted that machines as intelligent as humans would exist within a generation. "
    "Optimism was widespread, but progress was slower than expected."
]

ppl = compute_perplexity(model, tokenizer, texts, device=DEVICE, max_length=1024)
print(f"compute_perplexity() result: {ppl:.2f}")
print("(Expected range for GPT-2 on clean English text: 20-60)")

## 3. Validate `full_cache.py` — Full KV Cache Baseline (Stage 1)

This is the **standard HuggingFace decoding** with no modifications — the KV cache grows unbounded as new tokens are generated.

This is our primary reference point. Every other method will be compared against these numbers.
A good method should match this perplexity while using significantly less memory.

In [ ]:
from src.baseline.full_cache import run_full_cache

result = run_full_cache(model, tokenizer, "Once upon a time", max_new_tokens=50, device=DEVICE)

print("run_full_cache() result:")
print(f"  generated_text        : {result['generated_text']}")
print(f"  latency_ms_per_token  : {round(result['latency_ms_per_token'], 2)} ms/token")
print(f"  throughput_tokens_per_sec: {round(result['throughput_tokens_per_sec'], 1)} tok/s")
print(f"  peak_memory_gb        : {round(result['peak_memory_gb'], 3)} GB")

## 4. Stage 1 Benchmark — Full Cache on WikiText-103

Run the full benchmark pipeline for the full cache baseline.

**Dataset:** WikiText-103 — Wikipedia articles, the standard perplexity benchmark for LLMs.
`num_samples=5` here for quick validation. Official experiments use 50–100 samples.

In [ ]:
from src.eval.benchmark import run_benchmark, save_results

config = {
    "method": "full_cache",
    "model": MODEL_NAME,
    "dataset": "wikitext-103",
    "num_samples": 5,
    "max_new_tokens": 50,
    "context_len": 1024,
}

results = run_benchmark(config, model, tokenizer)

print("\n=== Benchmark Results ===")
for k, v in results.items():
    print(f"  {k}: {v}")

save_results(results)

---

# Stage 2 — Simple Baselines

Stage 1 established the full cache as our reference point.
Stage 2 implements two simpler eviction strategies that trade generation quality for memory savings.

These are **known methods from prior work** — we use them as lower-bound baselines to show that naive approaches hurt quality, which motivates our adaptive method in Stage 3.

| Method | Key idea | Prior work |
|--------|----------|-----------|
| Sliding window | Keep only the last N tokens | StreamingLLM (Xiao et al., 2023) |
| Naive truncation | Hard-cut prompt to fixed length | Standard baseline in most KV cache papers |

## 5. Validate `sliding_window.py` — Sliding Window Baseline (Stage 2)

Implements the **StreamingLLM** eviction strategy: keep only the most recent `window_size` tokens in the KV cache.

**How it works:** We write a manual decoding loop instead of using `model.generate()`.
After each token is generated, we check if `past_key_values` exceeds `window_size`.
If it does, we slice off the oldest tokens: `k[:, :, -window_size:, :]`.

**Expected tradeoff:** Lower memory than full cache, but higher perplexity on long documents
because the model loses access to important context from earlier in the sequence.

In [ ]:
from src.baseline.sliding_window import run_sliding_window

result_sw = run_sliding_window(
    model, tokenizer, "Once upon a time",
    window_size=32,
    max_new_tokens=50,
    device=DEVICE,
)

print("run_sliding_window() result:")
print(f"  generated_text        : {result_sw['generated_text']}")
print(f"  latency_ms_per_token  : {round(result_sw['latency_ms_per_token'], 2)} ms/token")
print(f"  throughput_tokens_per_sec: {round(result_sw['throughput_tokens_per_sec'], 1)} tok/s")
print(f"  peak_memory_gb        : {round(result_sw['peak_memory_gb'], 3)} GB")

## 6. Validate `naive_truncation.py` — Naive Truncation Baseline (Stage 2)

The simplest possible baseline: **truncate the prompt to `max_cache_size` tokens before generation begins**.
The KV cache is not managed during generation — it still grows unbounded after truncation.

**How it differs from sliding window:**
- Sliding window: dynamically evicts tokens *during* generation at every step
- Naive truncation: one-time hard cut *before* generation starts, no further management

**Expected tradeoff:** Worst perplexity of all methods on long documents, because the model
only ever sees a limited prefix of the input — any important context beyond `max_cache_size` is simply gone.

In [ ]:
from src.baseline.naive_truncation import run_naive_truncation

result_nt = run_naive_truncation(
    model, tokenizer, "Once upon a time",
    max_cache_size=16,
    max_new_tokens=50,
    device=DEVICE,
)

print("run_naive_truncation() result:")
print(f"  generated_text        : {result_nt['generated_text']}")
print(f"  latency_ms_per_token  : {round(result_nt['latency_ms_per_token'], 2)} ms/token")
print(f"  throughput_tokens_per_sec: {round(result_nt['throughput_tokens_per_sec'], 1)} tok/s")
print(f"  peak_memory_gb        : {round(result_nt['peak_memory_gb'], 3)} GB")

## 7. Stage 2 Benchmark — All Baselines Side by Side

Run all three baselines on WikiText-103 and compare results in one table.

This is the core output of Stage 1 & 2. The numbers here become the reference point for Stage 3,
where our adaptive method should achieve similar memory savings but with much smaller perplexity increase.

In [ ]:
import pandas as pd
from src.eval.benchmark import run_benchmark, save_results

base_config = {
    "model": MODEL_NAME,
    "dataset": "wikitext-103",
    "num_samples": 5,
    "max_new_tokens": 50,
    "context_len": 1024,
}

configs = [
    {**base_config, "method": "full_cache"},
    {**base_config, "method": "sliding_window", "method_kwargs": {"window_size": 32}},
    {**base_config, "method": "naive_truncation", "method_kwargs": {"max_cache_size": 256}},
]

all_results = []
for config in configs:
    print(f"\nRunning: {config['method']}")
    results = run_benchmark(config, model, tokenizer)
    save_results(results)
    all_results.append(results)

df = pd.DataFrame(all_results)[["method", "avg_latency_ms_per_token", "avg_throughput_tokens_per_sec", "avg_peak_memory_gb", "perplexity"]]
display(df)

## 8. Results Summary

Load all saved CSVs from `experiments/results/` and display a combined table.

This aggregates results across all runs. When we later add the adaptive method (Stage 3),
its row will appear here automatically alongside the baselines.